# Characterization of loop dynamics in kinases

We present a workflow to discover protein conformational features associated with kinase loop rearrangments. The purpose of this notebook is to describe the necessary steps adopted in our study. Implementations of the described steps are included as `.py` files within the folder `workflow`.

## Table of contents

This modelling pipeline is subdivided in the following sections:

4. [Feature definition](#4)
   1. [Feature matrix](#41)
5. [Feature selection](#5)

In this notebook, we will be focusing on the second step: feature selection.

![State of the workflow](images/FeatureSelection.png)

To get started, let's load some packages!

In [ ]:
# File and system operations
import os
import sys
import subprocess
from glob import glob
import pickle
import shutil

# Data processing
import pandas as pd
import numpy as np
import mdtraj as md

# Network and parallel processing
import requests
import time
import multiprocessing
import concurrent.futures

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# custom utility functions and class
from workflow.utilities import count_pdb_files, braf_res, clear_and_make, make_seg, copy_filtered_pdbs
from workflow.utilities import PDBDownloader

# 5. Feature selection  <a id="5"></a>

In this section we will study how each feature is distributed across our kinase dataset and we will filter out features that are not statistically relevant in order to facilitate the classification step.

If you've already computed feature matrices and saved them, you can skip the previous cells and reload the data here using the class `FeatureSelection`.

In [ ]:
from workflow.feature_selection import FeatureSelection

# Create new FeatureSelection object
fs = FeatureSelection(dfg_index=145, ape_index=174, conservation_threshold=0.97)

# Load reference data
fs.load_results('reference_data.pkl')

# Load feature matrix
feature_df = pd.read_csv('feature_matrix.csv', index_col=0)
fs.feature_matrix = feature_df.values
fs.structure_names = list(feature_df.index)

# Load labels
labels_df = pd.read_csv('labels.csv')
fs.labels = labels_df['label'].values

# Load distance dataframe
fs.intra_structure_df = pd.read_csv('intra_structure_distances.csv')

# Calculate statistics
final_shape = fs.feature_matrix.shape
final_total = fs.feature_matrix.size
final_valid = np.sum(~np.isnan(fs.feature_matrix))

print(f"\n✅ Reloaded successfully!")
print(f"\n📊 Feature matrix:")
print(f"   Matrix shape: {final_shape[0]:,} structures × {final_shape[1]:,} residue pairs")
print(f"   Total entries: {final_total:,}")
print(f"   Valid measurements: {final_valid:,}")
print(f"   NaN values: {final_total - final_valid:,}")


We first look for abnormally large features that might indicate structural issues and exclude them from the feature matrices. 

In [ ]:
# Check prerequisites
try:
    fs
    if fs.feature_matrix is None:
        raise ValueError("Feature matrix not built yet. Run Cell 113 first.")
except NameError:
    raise NameError("FeatureSelection object 'fs' not defined. Run Cell 102 first.")

# One-liner replacement for the long outlier/NaN-cleaning snippet.
# - sets >50Å distances to NaN
# - drops all-NaN features/structures
# - saves outlier table to CSV
results = fs.filter_outlier_distances_and_drop_nan(
    threshold=50.0,
    set_to_nan=True,
    max_nan_fraction=1.0,
    outliers_csv_path="outlier_distances.csv",
    print_top_n=10,
    verbose=True,
)


We then specifically look for highly-correlated features, construct feature classess of correlated features and pick a representative feature from each based on the largest variance.

In [ ]:
# Correlation-based feature selection
# Identify groups of highly correlated features and keep only the feature
# with the highest standard deviation from each group

print("\n" + "="*60)
print("CORRELATION-BASED FEATURE SELECTION")
print("="*60)

# Speed tips:
# - If you have no NaN values, correlation will be much faster (uses numpy's corrcoef)
# - use_parallel=True enables parallel processing (2-8x faster with NaN values)
# - Set plot_histogram=False to skip plotting
# - Increase correlation_threshold (e.g., 0.95) to find fewer groups

print(f"Current feature matrix shape: {fs.feature_matrix.shape}")
print(f"Has NaN values: {np.any(np.isnan(fs.feature_matrix))}")

# Perform correlation analysis
# If parallel processing has issues, set use_parallel=False to use the safe sequential method
selected_features, analysis_info = fs.filter_correlated_features(
    correlation_threshold=0.90,  # Higher threshold = fewer correlated groups = faster
    plot_histogram=True,          # Set to False to skip plotting
    plot_network=False,           # Set to True to see the correlation network (slow for many features)
    use_parallel=True,            # Set to False if you encounter issues with parallel processing
    n_jobs=-1                     # Use all CPU cores (-1), or specify number (e.g., 4)
)

# Apply the selection to the feature matrix
fs.apply_feature_selection(selected_features)

print("\n✅ Correlation-based feature selection complete!")

# Save intermediate results (after correlation selection)
print("\n" + "="*60)
print("Saving correlation-filtered feature matrix...")
print("="*60)
fs.save_results(output_prefix="corr_filtered_")
print("\n✅ Saved correlation-filtered results!")


If you've already run correlation-based feature selection and want to skip directly to same-mean and variance filtering, you can run the following cell.

In [ ]:
fs = FeatureSelection(dfg_index=145, ape_index=174, conservation_threshold=0.97)
fs.load_results('corr_filtered_reference_data.pkl')

feature_df = pd.read_csv('corr_filtered_feature_matrix.csv', index_col=0)
fs.feature_matrix = feature_df.values
fs.structure_names = list(feature_df.index)

labels_df = pd.read_csv('corr_filtered_labels.csv')
fs.labels = labels_df['label'].values

fs.intra_structure_df = pd.read_csv('corr_filtered_intra_structure_distances.csv')

print(f"✅ Reloaded correlation-filtered data!")
print(f"   Matrix shape: {fs.feature_matrix.shape}")
print(f"   Number of features: {len(fs.unique_pairs)}")


We exploit the ANOVA SUM method to select a sub-set of statistically-relevant features.

In [ ]:
# Filter features using ANOVA F-value
# Keeps top N features that best distinguish between classes (active vs inactive)

selected_anova_indices = fs.filter_anova_features(
    n_features=300,      # Keep top 300 features
    plot_scores=False,   # Set to True to see F-value distribution
    remove=True
)

print(f"\n✅ ANOVA F-value filtering complete!")
print(f"   Final feature count: {len(fs.unique_pairs)}")


In [ ]:
# Impute any remaining NaN values before saving and classification
fs.impute_remaining_nan(strategy='median')


In [ ]:
# Save the fully filtered feature matrix and related data
print("\n" + "="*60)
print("SAVING FILTERED FEATURE MATRIX")
print("="*60)

print(f"\nFinal feature matrix shape: {fs.feature_matrix.shape}")
print(f"  Structures: {fs.feature_matrix.shape[0]}")
print(f"  Features: {fs.feature_matrix.shape[1]}")

fs.save_results(output_prefix="filtered_")
print("\n✅ Saved filtered results! Can be reloaded for downstream analysis.")

# Print filtering summary
print("\n" + "="*60)
print("FILTERING SUMMARY")
print("="*60)
print("Applied filters in order:")
print("  1. ✓ Outlier distances (>50Å)")
print("  2. ✓ All-NaN features/structures")
print("  3. ✓ Correlation-based selection (r > 0.90)")
print("  4. ✓ ANOVA F-value selection (top 300)")
print(f"\nFinal: {fs.feature_matrix.shape[0]} structures × {fs.feature_matrix.shape[1]} features")
